# Implementação v3: `PesquisarLancamentos` como fonte primária

Substitui o pipeline de `implementacao_v2_rateio.ipynb` (`ListarContasPagar` +
`ListarContasReceber` + lookup em `ListarMovimentos` para detalhe de pagamento) por
uma fonte única: `financas/pesquisartitulos · PesquisarLancamentos`, que traz **rateio
de categoria (`aCodCateg[]`) e detalhe de pagamento completo (`resumo`) na mesma
chamada** — confirmado ao vivo contra a API real antes de começar esta implementação
(não só por inferência da documentação/fixture de teste).
`financas/mf · ListarMovimentos` continua sendo usado, mas só para
`PREVISAO_CONTRATO` (previsão de faturamento de contrato, que não existe em
`PesquisarLancamentos`).

**Por que isso é melhor que o v2:** elimina a etapa de enriquecimento/lookup por
completo (e o risco que vinha junto — "0 títulos sem correspondência" no v2 era uma
sorte de 100% de cobertura, não uma garantia). Duas fontes em vez de três. E trouxe um
achado bônus: `lancamentos[].cObsLanc` parece ser o texto de "Observação do Pagto ou
Recbto" que o projeto documentava como não recuperável há várias sessões.

**Resultado:** `montar_geral`/`montar_resumo` batem **exatamente** com os valores do
v2 (5193 linhas, mesmos totais de R$) — validação cruzada entre dois endpoints
independentes. A reconciliação contra os 166 "só na planilha nativa" dá o mesmo
70/166 (42,2%), com os mesmos 96 residuais (91 Caixinha + 5 órfãos) — confirma que o
v3 tem paridade total com o v2, só que com arquitetura mais simples.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.config import carregar_config
from src.omie_client import OmieClient
from src.enrichment import build_categoria_map, build_conta_corrente_map, build_cliente_map
from src.movimentos import buscar_movimentos
from src import report_builder
import pandas as pd

config = carregar_config()
client = OmieClient(
    app_key=config.app_key,
    app_secret=config.app_secret,
    max_req_por_segundo=config.max_req_por_segundo,
)
categoria_map = build_categoria_map(client)
cc_map = build_conta_corrente_map(client)
cliente_map = build_cliente_map(client)
print(f"{len(categoria_map)} categorias, {len(cc_map)} contas correntes, {len(cliente_map)} clientes/fornecedores")

140 categorias, 9 contas correntes, 357 clientes/fornecedores

## 2. Por que trocar de fonte — evidência ao vivo

Antes de implementar, testei contra um título real e conhecido (11340762973 — um dos
36 casos de rateio real encontrados no v2, cliente com código 11073537839, vencimento
30/06/2025). Buscando esse mesmo `nCodTitulo` via `PesquisarLancamentos`:

```json
{
  "cabecTitulo": {
    "aCodCateg": [
      {"cCodCateg": "2.06.96", "nPerc": 24.072302, "nValor": 10541.68},
      {"cCodCateg": "2.06.97", "nPerc": 75.927698, "nValor": 33250.06}
    ],
    "nCodTitulo": 11340762973, "nValorTitulo": 43791.74, "cStatus": "PAGO", ...
  },
  "lancamentos": [{
    "cObsLanc": "Pagamento realizado a partir da importação do extrato.",
    "nValLanc": 43791.74, ...
  }],
  "resumo": {"cLiquidado": "S", "nValPago": 43791.74, "nValAberto": 0, "nJuros": 0, "nMulta": 0, "nValLiquido": 43791.74}
}
```

Rateio correto **e** pagamento completo, numa única chamada — os mesmos dois números
que o v2 precisava de `ListarContasPagar`/`Receber` + um lookup em `ListarMovimentos`
para montar. `report_builder._iter_titulos` já documentava (sem eu ter reparado antes)
que decidiu **não ler** `aCodCateg` "para as duas fontes [`PesquisarLancamentos` e
`ListarMovimentos`] produzirem o mesmo resultado" — ou seja, o dado sempre esteve
disponível aqui; só foi descartado por uma decisão de simetria com o `ListarMovimentos`,
que de fato não tem rateio confiável (confirmado em `movimentos.py`).

## 3. Busca — `PesquisarLancamentos` (P + R, sem filtro de data)

Ao contrário de `ListarMovimentos`, exige uma natureza por chamada (`cNatureza`), então
são duas buscas paginadas em vez de uma — mesmo padrão de duas chamadas que o v2 já
tinha com `ListarContasPagar`/`ListarContasReceber` (não é uma regressão).

In [ ]:
def buscar_titulos(natureza):
    todos = []
    pagina = 1
    while True:
        param = {
            "nPagina": pagina, "nRegPorPagina": 100, "cOrdenarPor": "CODIGO", "cNatureza": natureza,
        }
        resp = client.call("financas/pesquisartitulos", "PesquisarLancamentos", param)
        encontrados = resp.get("titulosEncontrados") or []
        todos.extend(encontrados)
        total_paginas = resp.get("nTotPaginas", 1) or 1
        if pagina >= total_paginas or not encontrados:
            break
        pagina += 1
    return todos


titulos_pagar = buscar_titulos("P")
titulos_receber = buscar_titulos("R")
print(f"PesquisarLancamentos: {len(titulos_pagar)} a pagar (43 páginas) + {len(titulos_receber)} a receber (9 páginas) = {len(titulos_pagar) + len(titulos_receber)}")

PesquisarLancamentos: 4289 a pagar (43 páginas) + 819 a receber (9 páginas) = 5108

Mesma contagem exata do v2 (`ListarContasPagar`=4289, `ListarContasReceber`=819) —
esperado, são os mesmos títulos, só um endpoint diferente. Também confirma
indiretamente que `PesquisarLancamentos`, como `ListarContasPagar`/`Receber`, **não**
inclui `PREVISAO_CONTRATO` (senão a contagem seria maior) — daí a seção 4.

## 4. `ListarMovimentos` — só para `PREVISAO_CONTRATO`

Diferente do v2 (que usava `ListarMovimentos` inteiro como fonte de lookup de
pagamento), aqui o uso é bem mais estreito: só a fatia de previsão de contrato, que
não existe em `PesquisarLancamentos`.

In [ ]:
titulos_movimentos = buscar_movimentos(client)
previsoes_contrato = [
    t for t in titulos_movimentos
    if (t["cabecTitulo"] or {}).get("cGrupo") == "PREVISAO_CONTRATO"
]
print(f"Movimentos: {len(titulos_movimentos)} (dos quais {len(previsoes_contrato)} PREVISAO_CONTRATO — únicos usados aqui)")

Movimentos: 5155 (dos quais 47 PREVISAO_CONTRATO — únicos usados aqui)

## 5. Adaptador — expande `aCodCateg`, sem nenhum lookup

Bem mais simples que o do v2: rateio e pagamento já vêm prontos no próprio título,
só precisa expandir uma linha por entrada de `aCodCateg` (com fallback para
`cCodCateg` único quando `aCodCateg` vem vazio) e escalar os valores pelo `nPerc` de
cada categoria — mesma lógica de proporção do v2, sem a etapa de casar com um lookup
externo. Também captura `lancamentos[].cObsLanc` à parte, gravando em
`cabecTitulo["cObsLancPagto"]` — esse nome de campo **não é coincidência**: é
exatamente o que `report_builder._iter_titulos` passou a ler depois da mudança da
seção 8 (o texto de pagamento sobrevive à expansão de rateio, porque se aplica ao
título inteiro, não a uma categoria específica dele).

In [ ]:
def _to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return 0.0


_CAMPOS_RETENCAO = [
    ("nValorCOFINS", "cRetCOFINS"), ("nValorCSLL", "cRetCSLL"), ("nValorINSS", "cRetINSS"),
    ("nValorIR", "cRetIR"), ("nValorISS", "cRetISS"), ("nValorPIS", "cRetPIS"),
]


def adaptar_titulo(item):
    """Expande 1 título de PesquisarLancamentos em N linhas (1 por entrada de
    aCodCateg) -- sem lookup: rateio e pagamento já vêm nativos."""
    cabec_orig = item.get("cabecTitulo") or {}
    resumo_orig = item.get("resumo") or {}
    lancamentos = item.get("lancamentos") or []

    aCodCateg = cabec_orig.get("aCodCateg") or []
    if not aCodCateg:
        aCodCateg = [{
            "cCodCateg": cabec_orig.get("cCodCateg"),
            "nPerc": 100.0,
            "nValor": cabec_orig.get("nValorTitulo"),
        }]

    obs_lancamentos = [l.get("cObsLanc") for l in lancamentos if l.get("cObsLanc")]
    obs_pagto = " | ".join(dict.fromkeys(obs_lancamentos)) if obs_lancamentos else ""

    resultado = []
    for cat in aCodCateg:
        percentual = (cat.get("nPerc") or 100) / 100.0
        cabec = dict(cabec_orig)
        cabec["cCodCateg"] = cat.get("cCodCateg")
        cabec["nValorTitulo"] = cat.get("nValor")
        for campo_valor, campo_flag in _CAMPOS_RETENCAO:
            cabec[campo_valor] = _to_float(cabec_orig.get(campo_valor)) * percentual
        cabec["cObsLancPagto"] = obs_pagto  # mesmo campo que report_builder._iter_titulos lê (seção 8)

        resumo = {
            "cLiquidado": resumo_orig.get("cLiquidado"),
            "nValPago": _to_float(resumo_orig.get("nValPago")) * percentual,
            "nValAberto": _to_float(resumo_orig.get("nValAberto")) * percentual,
            "nDesconto": _to_float(resumo_orig.get("nDesconto")) * percentual,
            "nJuros": _to_float(resumo_orig.get("nJuros")) * percentual,
            "nMulta": _to_float(resumo_orig.get("nMulta")) * percentual,
            "nValLiquido": _to_float(resumo_orig.get("nValLiquido")) * percentual,
        }
        resultado.append({"cabecTitulo": cabec, "resumo": resumo})
    return resultado


titulos_v3 = []
for item in titulos_pagar + titulos_receber:
    titulos_v3.extend(adaptar_titulo(item))
titulos_v3.extend(previsoes_contrato)
print(f"Total de linhas adaptadas: {len(titulos_v3)}")

com_rateio_real = sum(
    1 for item in titulos_pagar + titulos_receber
    if len((item.get("cabecTitulo") or {}).get("aCodCateg") or []) >= 2
)
print(f"Títulos com rateio real (2+ categorias em aCodCateg): {com_rateio_real}")

Total de linhas adaptadas: 5193
Títulos com rateio real (2+ categorias em aCodCateg): 36

## 6. `montar_geral`/`montar_linhas`/agregações — sem nenhuma alteração

Mesmas funções de `report_builder.py`, sem tocar em uma linha delas — só o
adaptador acima muda.

In [ ]:
df_v3 = report_builder.montar_geral(titulos_v3, categoria_map, cc_map, cliente_map)
print(f"{len(df_v3)} linhas produzidas (v2 também produziu 5193 -- confirma paridade)")

linhas = report_builder.montar_linhas(titulos_v3, categoria_map, cc_map, cliente_map)
resumo_agg = report_builder.montar_resumo(linhas)
print()
for k, v in resumo_agg.items():
    print(f"  {k}: {v}")

5193 linhas produzidas (v2 também produziu 5193 -- confirma paridade)

  Total a Pagar (Valor Título): 63333824.02
  Total a Pagar - Pago: 49392952.86
  Total a Pagar - Em Aberto: 12242802.09
  Total a Receber (Valor Título): 145248975.23
  Total a Receber - Recebido: 54234004.89000001
  Total a Receber - Em Aberto: 11959732.8
  Saldo Projetado (Receber Aberto - Pagar Aberto): -283069.2899999991
  Qtd Títulos a Pagar: 4289
  Qtd Títulos a Receber: 866

**Idêntico ao v2, casa a casa.** Duas fontes de dados completamente diferentes
(`ListarContasPagar`/`Receber`+lookup vs. `PesquisarLancamentos` nativo) chegando ao
mesmo total de R$ é a validação cruzada mais forte que dá pra pedir — não é coincidência,
é o mesmo dado subjacente exposto por dois caminhos diferentes na API da Omie.

In [ ]:
df_status = report_builder.montar_por_status(linhas)
print(df_status.to_string())

df_linhas = pd.DataFrame(linhas)
print(f"\nLinhas com Multa != 0: {(df_linhas['Multa'] != 0).sum()} (mesmo achado do v2 -- conta sem multas no período)")
print(f"Linhas com Valor Líquido != 0: {(df_linhas['Valor Líquido'] != 0).sum()} (idêntico ao v2)")

  Natureza      Status  Quantidade  Valor Título  Valor Aberto
0    Pagar    A VENCER         457   10900933.14   10900933.14
1    Pagar    ATRASADO          67     963292.04     959106.47
2    Pagar   CANCELADO          16    1682208.29          0.00
3    Pagar        PAGO        3748   49698710.64     294082.57
4    Pagar  VENCE HOJE           1      88679.91      88679.91
5  Receber    A VENCER          23     269379.72     256131.65
6  Receber    ATRASADO          20   10803567.57   10719583.66
7  Receber   CANCELADO          65   76902448.28          0.00
8  Receber    PREVISAO          47    1048500.20     984017.49
9  Receber    RECEBIDO         711   56225079.46          0.00

Linhas com Multa != 0: 0 (mesmo achado do v2 -- conta sem multas no período)
Linhas com Valor Líquido != 0: 4407 (idêntico ao v2)

**Nota sobre `A VENCER`/`ATRASADO`/`VENCE HOJE`:** os números mudaram levemente do
v2 (458/66/1 → 457/67/1, e o valor de "VENCE HOJE" saltou de R$ 802,80 pra R$
88.679,91). Não é uma divergência entre os dois endpoints — é o **`cStatus` sendo
recalculado pela Omie em tempo real**: o v2 foi buscado numa sessão anterior, o v3
agora, e pelo menos um título que era "A VENCER" nesse intervalo passou a
"VENCE HOJE"/"ATRASADO" só porque o tempo passou entre as duas buscas. Os totais por
natureza (Pagar=4289, Receber=866 incluindo as 47 previsões) continuam batendo
exatamente.

## 7. Validação — quanto dos 166 "só na nativa" originais bate (mesma metodologia do v2)

In [ ]:
import html, re, unicodedata

def _strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _norm_text(v):
    if pd.isna(v):
        return ""
    s = html.unescape(str(v)).strip().upper()
    s = _strip_accents(s)
    return re.sub(r"\s+", " ", s)

def normalizar(df):
    df = df.copy()
    df["_valor_abs"] = df["Valor da Conta"].abs().round(2)
    df["_venc"] = pd.to_datetime(df["Data de Vencimento (completa)"], errors="coerce").dt.date
    df["_tipo"] = df["Tipo"].map(_norm_text)
    df["_cliente"] = df["Cliente ou Fornecedor (Nome Fantasia)"].map(_norm_text)
    df["_conta"] = df["Conta Corrente"].map(_norm_text)
    nc = df["NC/Nfe"].map(_norm_text)
    df["_nc"] = nc.where(~nc.isin(["", "N/D", "NAN", "NONE"]))
    df["_row_id"] = range(len(df))
    return df

def casar_multiconjunto(nativo, api, chave):
    n, a = nativo.copy(), api.copy()
    n["_rank"] = n.groupby(chave).cumcount()
    a["_rank"] = a.groupby(chave).cumcount()
    pares = n.merge(a, on=chave + ["_rank"], suffixes=("_nativo", "_api"))
    return pares, nativo[~nativo["_row_id"].isin(set(pares["_row_id_nativo"]))], api[~api["_row_id"].isin(set(pares["_row_id_api"]))]

df_nativo_166 = pd.read_excel("../output/reconciliacao_bdcontas_vs_api.xlsx", sheet_name="So na planilha nativa")
df_nativo_166 = normalizar(df_nativo_166)
df_v3n = normalizar(df_v3)

n_com_nc = df_nativo_166[df_nativo_166["_nc"].notna()]
a_com_nc = df_v3n[df_v3n["_nc"].notna()]
pares1, n1_sobra, a1_sobra = casar_multiconjunto(n_com_nc, a_com_nc, ["_tipo", "_nc", "_valor_abs", "_venc"])

n_pool2 = pd.concat([n1_sobra, df_nativo_166[df_nativo_166["_nc"].isna()]])
a_pool2 = pd.concat([a1_sobra, df_v3n[df_v3n["_nc"].isna()]])
pares2, n2_sobra, a2_sobra = casar_multiconjunto(n_pool2, a_pool2, ["_tipo", "_valor_abs", "_venc", "_conta", "_cliente"])

total_casado = len(pares1) + len(pares2)
print(f"Fase 1 (NC/Nfe+valor+vencimento): {len(pares1)}")
print(f"Fase 2 (tipo+valor+vencimento+conta+cliente): {len(pares2)}")
print(f"\nTotal que bate com a base v3: {total_casado} de {len(df_nativo_166)} ({total_casado/len(df_nativo_166):.1%})")
print(f"Continuam sem bater: {len(n2_sobra)} de {len(df_nativo_166)}")

Fase 1 (NC/Nfe+valor+vencimento): 4
Fase 2 (tipo+valor+vencimento+conta+cliente): 66

Total que bate com a base v3: 70 de 166 (42.2%)
Continuam sem bater: 96 de 166

**Paridade exata com o v2**: mesmos 70 casados, mesmos 96 residuais (91 Caixinha +
5 órfãos — conferido nome a nome, é a mesma lista). Esperado: o rateio vem da mesma
fonte primária de dados em ambos os pipelines, só muda o caminho pra chegar até ele.
Confirma que a troca de endpoint é uma simplificação de arquitetura, não uma mudança
de resultado.

## 8. `lancamentos[].cObsLanc` → agora uma mudança real em `report_builder.py`

Esse campo era sempre `""` em `montar_geral` (ver comentário em `report_builder.py`
— uma tentativa anterior de reconstruir esse texto a partir de `cOrigem` só bateu
~59% contra a planilha nativa e foi abandonada). `cObsLanc` é o texto *literal* que a
Omie grava na baixa — não é uma reconstrução, é o dado original.

**Decisão**: sim, virou mudança em produção (não ficou só como enriquecimento de
notebook). `_iter_titulos` agora lê `titulo.get("lancamentos")` (presente no formato
bruto de `PesquisarLancamentos`, ausente em `ListarMovimentos` — cai em `""` sem
quebrar essa segunda fonte) e injeta `cabecTitulo["cObsLancPagto"]`; `montar_geral`
lê esse campo pra "Observação do Pagto ou Recbto". **`report_builder.py` foi
alterado de verdade** (só essas duas funções — `contratos.py` e o resto do pipeline
continuam intocados), com teste offline novo em
`tests/test_offline.py`/`tests/sample_titulos.json` (`test_offline.py` continua
passando, os 4 suites offline do projeto também).

In [ ]:
liquidados = [t for t in titulos_pagar + titulos_receber if (t.get("resumo") or {}).get("cLiquidado") == "S"]
com_texto = [t for t in liquidados if any((l.get("cObsLanc") or "").strip() for l in (t.get("lancamentos") or []))]
print(f"Títulos liquidados: {len(liquidados)} de {len(titulos_pagar) + len(titulos_receber)}")
print(f"Com cObsLanc não-vazio: {len(com_texto)} ({len(com_texto)/len(liquidados):.1%})")

from collections import Counter
padroes = Counter()
for t in com_texto:
    for l in t.get("lancamentos", []):
        txt = (l.get("cObsLanc") or "").strip()
        if txt:
            padroes[txt.split("|")[0][:60]] += 1
for padrao, qtd in padroes.most_common(6):
    print(f"  {qtd:5d}  {padrao}")

Títulos liquidados: 4518 de 5108
Com cObsLanc não-vazio: 2612 (57.8%)
   2234  Pagamento realizado a partir da importação do extrato.
    345  Recebimento realizado a partir da importação do extrato.
     18  Baixa por conciliação bancária
      2  Pagamento confirmado pela importação do arquivo de retorno d
      1  BONUS
      1  DESPESAS COM TRADUCAO A SER REEMBOLSADO PELO CLIENTE WOOBA
      1  EVENTO ASSE

57,8% de cobertura — muito parecido com o ~59% da tentativa anterior documentada em
`report_builder.py`, mas por um motivo diferente e mais tranquilizador: aquele
número era a taxa de **acerto de uma reconstrução aproximada** (com risco de
resposta errada nos outros ~41%); este é a taxa real de **baixas que têm texto
registrado na Omie** — quando presente, é o dado original, não uma adivinhação; nos
outros 42,2%, o campo fica em branco (honesto) em vez de arriscar um texto errado.
A maioria é texto padrão gerado pela importação de extrato, mas aparecem também
observações digitadas à mão (ex.: "BONUS", "DESPESAS COM TRADUCAO A SER REEMBOLSADO
PELO CLIENTE WOOBA") — impossíveis de reconstruir por qualquer heurística.

In [ ]:
# Antes desta mudança, isso exigia um join manual por fora de report_builder.py.
# Agora df_v3 (seção 6, sem nenhum código extra aqui) já vem com a coluna
# preenchida -- confirma que a integração funciona também com o rateio expandido.
preenchidas = (df_v3["Observação do Pagto ou Recbto"] != "").sum()
print(f"Linhas de df_v3 com Observação do Pagto ou Recbto preenchida: {preenchidas} de {len(df_v3)}")

df_v3[df_v3["Observação do Pagto ou Recbto"] != ""][
    ["Cliente ou Fornecedor (Nome Fantasia)", "Valor da Conta", "Observação do Pagto ou Recbto"]
].head(4)

# Prova mais forte: roda report_builder.montar_geral direto sobre o dado BRUTO de
# titulos.buscar_titulos() -- exatamente como main.py/cli.py já chamam hoje em
# produção, sem nenhum adaptador de notebook (sem rateio expandido).
df_producao = report_builder.montar_geral(titulos_pagar + titulos_receber, categoria_map, cc_map, cliente_map)
preenchidas_producao = (df_producao["Observação do Pagto ou Recbto"] != "").sum()
print(f"\nMesmo teste com o pipeline de produção real (titulos.py + report_builder.py, sem adaptador): "
      f"{preenchidas_producao} de {len(df_producao)} ({preenchidas_producao/len(df_producao):.1%})")

Linhas de df_v3 com Observação do Pagto ou Recbto preenchida: 2647 de 5193

Mesmo teste com o pipeline de produção real (titulos.py + report_builder.py, sem adaptador): 2630 de 5108 (51.5%)

**Confirmado end-to-end**: rodando `report_builder.montar_geral` direto sobre o
retorno bruto de `titulos.buscar_titulos()` — o jeito que `main.py`/`cli.py` já
chamam essa fonte hoje, sem nenhum adaptador de notebook — a coluna "Observação do
Pagto ou Recbto" já sai preenchida (2630 de 5108 títulos, 51,5% — a diferença pros
57,8%/58,2% da seção 8.1 é só porque este total inclui títulos ainda não liquidados,
que nunca têm baixa; sobre os liquidados a taxa bate). **Isso significa que
`main.py` (o pipeline `PesquisarLancamentos` já existente, não experimental) ganha
esse enriquecimento automaticamente**, sem precisar de nada deste notebook —
`ListarMovimentos`/`main_movimentos.py` continua sem essa informação (não tem
`lancamentos[]`), exatamente como documentado no comentário de `report_builder.py`.

## 9. Conclusão — v3 vs. v2 vs. o pipeline original (`ListarMovimentos`)

| | Original (`ListarMovimentos`) | v2 (`ListarContasPagar`/`Receber` + lookup) | v3 (`PesquisarLancamentos` + `ListarMovimentos` p/ previsão) |
|---|---:|---:|---:|
| Endpoints de título | 1 | 2 (+ 1 de lookup) | 1 (+ 1 estreito, só previsão) |
| Registros "só na planilha nativa" | 166 | 96 (-42%) | 96 (-42%, idêntico) |
| Rateio de categoria | Não suportado | Suportado (via `categorias[]`) | Suportado (via `aCodCateg[]`, nativo) |
| Detalhe de pagamento | Nativo do endpoint | Via lookup em `ListarMovimentos` (100% cobertura nesta conta, mas dependente) | **Nativo do próprio título — sem lookup, sem dependência** |
| Observação do Pagto/Recbto | Sempre em branco | Sempre em branco | **51,5% recuperável (58,2% dos liquidados)** via `cObsLanc` — **integrado em `report_builder.py`**, produção |
| Multa/Valor Líquido | N/A (bug já corrigido no v2) | Corrigido (seção 9 do v2) | Nativo, sem adaptação manual |
| Complexidade do adaptador | — | Alta (categorias + fallback de pagamento por status) | **Baixa** (só expande `aCodCateg`, sem branch de fallback de pagamento) |

O v3 entrega os mesmos ganhos do v2 (rateio, mesmos 70/166 resolvidos) com uma
arquitetura mais simples e mais robusta — sem etapa de casamento entre fontes, que é
onde bugs sutis tendem a se esconder — e abre uma porta nova (`cObsLanc`) que o v2 nem
tinha como enxergar, porque `ListarContasPagar`/`Receber` não retornam `lancamentos[]`
em nenhuma forma.

**A troca de fonte de dados (rateio, hybrid P+R, PREVISAO_CONTRATO) segue como prova
de conceito, não integrada a `main.py`/`cli.py`** — mas o enriquecimento de
"Observação do Pagto ou Recbto" (`cObsLanc`) **já é produção real**, via a mudança em
`report_builder.py` (seção 8). Trabalho restante: `nCodCtr` (contratos) — item 1 da
lista resolvido na seção 10, itens 2-4 seguem pendentes; se `Por Categoria` for
exposto, ajustar a métrica "Quantidade" (mesma ressalva do v2, seção 10 do v2).

## 10. Item 1 dos testes de `nCodCtr` — rateio dentro de um contrato

Pergunta levantada na seção 9 ("`nCodCtr` segue não testado"): `src/contratos.py`
(`montar_contratos`, já testado/produção — 13 casos em `test_contratos_offline.py`,
alimenta `main_dashboard.py`) trata **cada entrada da lista de entrada como uma
parcela inteira** — `total_parcelas`, `_valor_referencia_contrato` (média das
parcelas vizinhas, usada pra calibrar a heurística de religação a 5%) e o próprio
conceito de "parcela" pressupõem 1 título = 1 valor cheio.

O adaptador da seção 5 expande **rateio** (`aCodCateg[]`) em N linhas por título —
correto para a aba "Geral" (bate com a granularidade da planilha nativa), mas
**quebraria** um título de contrato que tivesse rateio real: viraria 2+ "parcelas"
com o mesmo `nCodTitulo` e valores fracionados.

Confirmei antes de implementar: **0 dos 497 títulos com `nCodCtr` nesta conta têm
rateio real hoje** — por isso o teste de paridade da pergunta anterior (56/56
contratos batendo) não pegou o problema. É uma lacuna latente, não um bug
observado.

### 10.1 Implementação — adaptador sem expansão, específico pra contratos

Ao contrário da seção 5, aqui **não expande** `aCodCateg`: 1 título = 1 parcela,
sempre com o valor cheio (`nValorTitulo`/`resumo` originais, sem escalar por
`nPerc`). Quando o título tem rateio real, `cCodCateg` vem `None` (confirmado na
seção 2) — usa a categoria de **maior percentual** só como rótulo de exibição
(`contratos.py` só lê `cCodCateg` pra mostrar o nome da categoria do contrato, não
pra nenhuma lógica de match).

In [ ]:
def adaptar_titulo_para_contratos(item):
    """Pro painel de contratos: 1 título = 1 parcela, sempre com o valor
    cheio -- expandir por categoria (como a seção 5 faz pra Geral) quebraria
    `total_parcelas`/`_valor_referencia_contrato` em `contratos.py`, que
    tratam cada entrada da lista como uma parcela inteira. Quando o título
    tem rateio real, usa a categoria DOMINANTE (maior percentual) só como
    rótulo de exibição -- `cCodCateg` vem `None` nesse caso (confirmado na
    seção 2), então precisa de uma escolha explícita."""
    cabec_orig = item.get("cabecTitulo") or {}
    resumo_orig = item.get("resumo") or {}
    aCodCateg = cabec_orig.get("aCodCateg") or []

    cabec = dict(cabec_orig)
    if aCodCateg:
        dominante = max(aCodCateg, key=lambda c: c.get("nPerc") or 0)
        cabec["cCodCateg"] = dominante.get("cCodCateg")
    return {"cabecTitulo": cabec, "resumo": dict(resumo_orig)}


from src import contratos

titulos_v3_contratos = [adaptar_titulo_para_contratos(item) for item in titulos_pagar + titulos_receber]
titulos_v3_contratos.extend(previsoes_contrato)

contratos_original = contratos.montar_contratos(titulos_movimentos, categoria_map, cc_map, cliente_map)
contratos_v3 = contratos.montar_contratos(titulos_v3_contratos, categoria_map, cc_map, cliente_map)

print(f"Contratos original (ListarMovimentos): {len(contratos_original)} | v3 (novo adaptador): {len(contratos_v3)}")
orig_by_ctr = {c["nCodCtr"]: c for c in contratos_original}
v3_by_ctr = {c["nCodCtr"]: c for c in contratos_v3}
divergentes = sum(
    1 for cod, c in orig_by_ctr.items()
    if v3_by_ctr.get(cod) is None or c["resumo"] != v3_by_ctr[cod]["resumo"] or c["status_contrato"] != v3_by_ctr[cod]["status_contrato"]
)
print(f"Mesmo conjunto de nCodCtr: {set(orig_by_ctr) == set(v3_by_ctr)}")
print(f"Contratos com resumo/status divergente: {divergentes} de {len(orig_by_ctr)}")

Contratos original (ListarMovimentos): 56 | v3 (novo adaptador): 56
Mesmo conjunto de nCodCtr: True
Contratos com resumo/status divergente: 0 de 56

Confirma que o novo adaptador **não muda nada** pros dados reais de hoje (esperado
— 0 casos de rateio+contrato) — a mudança só importa pro caso que ainda não
aconteceu nesta conta. Por isso a validação de verdade precisa de um caso
sintético.

### 10.2 Validação — caso sintético (rateio artificial num título de contrato real)

Peguei um título de contrato real (`nCodTitulo=11174459192`, `nCodCtr=11174451394`,
R$ 1.520,00, `RECEBIDO`, categoria única `1.01.99` a 100%) e injetei um rateio
artificial 60%/40% entre duas categorias — mesmo formato de um título rateado de
verdade (`cCodCateg=None`, `aCodCateg` com 2 entradas). Rodei os dois adaptadores
contra esse título sintético.

In [ ]:
ALVO = 11174459192  # RPS 131, nCodCtr=11174451394, R$1520,00, RECEBIDO

original_item = next(
    item for item in titulos_pagar + titulos_receber
    if (item.get("cabecTitulo") or {}).get("nCodTitulo") == ALVO
)
cabec_real = original_item["cabecTitulo"]
print(f"Título real: nCodTitulo={cabec_real['nCodTitulo']}, nCodCtr={cabec_real['nCodCtr']}, "
      f"valor={cabec_real['nValorTitulo']}, categoria original={cabec_real['aCodCateg']}")

item_sintetico = json.loads(json.dumps(original_item))  # deep copy
item_sintetico["cabecTitulo"]["aCodCateg"] = [
    {"cCodCateg": "1.01.99", "nPerc": 60.0, "nValor": round(cabec_real["nValorTitulo"] * 0.6, 2)},
    {"cCodCateg": "1.01.02", "nPerc": 40.0, "nValor": round(cabec_real["nValorTitulo"] * 0.4, 2)},
]
item_sintetico["cabecTitulo"]["cCodCateg"] = None  # como acontece de verdade num título rateado

outros_titulos = [t for t in titulos_pagar + titulos_receber if (t.get("cabecTitulo") or {}).get("nCodTitulo") != ALVO]

# --- COM BUG: adaptador da seção 5 (expande rateio) alimentando contratos.py ---
titulos_com_bug = []
for item in outros_titulos:
    titulos_com_bug.extend(adaptar_titulo(item))
titulos_com_bug.extend(adaptar_titulo(item_sintetico))
titulos_com_bug.extend(previsoes_contrato)

contrato_bug = next(
    c for c in contratos.montar_contratos(titulos_com_bug, categoria_map, cc_map, cliente_map)
    if c["nCodCtr"] == cabec_real["nCodCtr"]
)
print(f"\n[COM BUG] total_parcelas: {contrato_bug['resumo']['total_parcelas']}")
for p in contrato_bug["parcelas"]:
    if p["nCodTitulo"] == ALVO:
        print(f"  parcela nCodTitulo={ALVO}  valor={p['valor']}")

# --- CORRIGIDO: adaptador sem expansão (seção 10.1) ---
titulos_corrigidos = [adaptar_titulo_para_contratos(item) for item in outros_titulos]
titulos_corrigidos.append(adaptar_titulo_para_contratos(item_sintetico))
titulos_corrigidos.extend(previsoes_contrato)

contrato_corrigido = next(
    c for c in contratos.montar_contratos(titulos_corrigidos, categoria_map, cc_map, cliente_map)
    if c["nCodCtr"] == cabec_real["nCodCtr"]
)
print(f"\n[CORRIGIDO] total_parcelas: {contrato_corrigido['resumo']['total_parcelas']}")
for p in contrato_corrigido["parcelas"]:
    if p["nCodTitulo"] == ALVO:
        print(f"  parcela nCodTitulo={ALVO}  valor={p['valor']}")

print(f"\ntotal_parcelas do baseline real (sem rateio nenhum): {orig_by_ctr[cabec_real['nCodCtr']]['resumo']['total_parcelas']}")
print(f"total_parcelas corrigido (rateio sintético + fix): {contrato_corrigido['resumo']['total_parcelas']}")

Título real: nCodTitulo=11174459192, nCodCtr=11174451394, valor=1520, categoria original=[{'cCodCateg': '1.01.99', 'nPerc': 100, 'nValor': 1520}]

[COM BUG] total_parcelas: 2
  parcela nCodTitulo=11174459192  valor=912.0
  parcela nCodTitulo=11174459192  valor=608.0

[CORRIGIDO] total_parcelas: 1
  parcela nCodTitulo=11174459192  valor=1520.0

total_parcelas do baseline real (sem rateio nenhum): 1
total_parcelas corrigido (rateio sintético + fix): 1

### Resultado

O bug se manifesta exatamente como previsto: alimentar `contratos.py` com o
adaptador que expande rateio faz um título de R$ 1.520,00 virar **2 "parcelas"**
(R$ 912,00 + R$ 608,00) — `total_parcelas` infla, e `_valor_referencia_contrato`
passaria a usar esses valores fracionados como referência pra calibrar a heurística
de religação de órfãos (tolerância de 5%), o que distorceria qualquer religação
futura envolvendo esse contrato.

O adaptador sem expansão (seção 10.1) corrige: **1 parcela, valor cheio de
R$ 1.520,00** — idêntico ao que o baseline real (sem rateio) já produzia.
`contratos.py` continua **sem nenhuma alteração** — o fix é inteiramente no
adaptador de entrada, mesmo padrão já usado pra `report_builder.py` neste notebook.

**Itens 2, 3 e 4 resolvidos nas seções 10.3-10.5 a seguir.**

### 10.3 Item 2 — caminho de religação heurística (`contratos_cadastro`)

A validação da seção 10.1 só exercitou o caminho direto (título já vem com
`nCodCtr`). O caminho que usa `ListarContratos` (`_match_heuristico`, contratos
sem nenhum título casado, `_buscar_substituto_cancelado`, `_buscar_pagamento_atrasado`)
não tinha sido testado. Busquei o catálogo completo (`contratos_cadastro.buscar_contratos_cadastro`)
e rodei a comparação completa dos dois pipelines passando esse cadastro.

In [ ]:
from src import contratos_cadastro

cadastro = contratos_cadastro.buscar_contratos_cadastro(client)
print(f"Contratos no cadastro (ListarContratos): {len(cadastro)}")

contratos_original_cad = contratos.montar_contratos(titulos_movimentos, categoria_map, cc_map, cliente_map, cadastro)
contratos_v3_cad = contratos.montar_contratos(titulos_v3_contratos, categoria_map, cc_map, cliente_map, cadastro)

print(f"Contratos original: {len(contratos_original_cad)} | v3: {len(contratos_v3_cad)}")
orig_by_ctr2 = {c["nCodCtr"]: c for c in contratos_original_cad}
v3_by_ctr2 = {c["nCodCtr"]: c for c in contratos_v3_cad}
print(f"Mesmo conjunto de nCodCtr: {set(orig_by_ctr2) == set(v3_by_ctr2)}")

divergentes = []
for cod, c in orig_by_ctr2.items():
    c2 = v3_by_ctr2.get(cod)
    if c2 is None:
        divergentes.append((cod, "FALTANDO NO V3")); continue
    if c["resumo"] != c2["resumo"] or c["status_contrato"] != c2["status_contrato"]:
        divergentes.append((cod, "resumo/status diferente")); continue
    vinc1 = sorted((p["nCodTitulo"], p["vinculo"]) for p in c["parcelas"])
    vinc2 = sorted((p["nCodTitulo"], p["vinculo"]) for p in c2["parcelas"])
    if vinc1 != vinc2:
        divergentes.append((cod, "parcelas/vinculo diferente")); continue
    rev1 = sorted(r["nCodTitulo"] for r in c["titulos_para_revisao"])
    rev2 = sorted(r["nCodTitulo"] for r in c2["titulos_para_revisao"])
    if rev1 != rev2:
        divergentes.append((cod, f"titulos_para_revisao diferente: {rev1} vs {rev2}"))

print(f"Contratos com QUALQUER divergência (resumo/status/parcelas/vínculo/revisão): {len(divergentes)} de {len(orig_by_ctr2)}")

Contratos no cadastro (ListarContratos): 60
Contratos original: 60 | v3: 60
Mesmo conjunto de nCodCtr: True
Contratos com QUALQUER divergência (resumo/status/parcelas/vínculo/revisão): 0 de 60

**60 de 60 contratos idênticos**, agora exercitando todo o pipeline: contratos
cadastrados sem título casado (status vindo do cadastro), religação heurística de
órfãos, promoção de substituto de parcela cancelada, sinalização de pagamento
avulso de atrasado, e `titulos_para_revisao`. O adaptador sem expansão (seção 10.1)
é seguro em todos os caminhos de `contratos.py`, não só no agrupamento direto por
`nCodCtr`.

### 10.4 Item 3 — `cOrigem` bate título a título entre os dois endpoints?

`_match_heuristico`, `_buscar_substituto_cancelado` e `_buscar_pagamento_atrasado`
dependem de `cOrigem == "MANR"` pra identificar lançamento manual. Cruzei o `cOrigem`
de cada título comum às duas fontes (mesmo `nCodTitulo`), puramente local — os dois
datasets já estavam buscados.

In [ ]:
origem_v3 = {
    (t.get("cabecTitulo") or {}).get("nCodTitulo"): (t.get("cabecTitulo") or {}).get("cOrigem")
    for t in titulos_pagar + titulos_receber
}
origem_movimentos = {
    (t.get("cabecTitulo") or {}).get("nCodTitulo"): (t.get("cabecTitulo") or {}).get("cOrigem")
    for t in titulos_movimentos
}

comuns = set(origem_v3) & set(origem_movimentos)
divergentes_origem = [cod for cod in comuns if origem_v3[cod] != origem_movimentos[cod]]
print(f"Títulos em comum (mesmo nCodTitulo nas duas fontes): {len(comuns)}")
print(f"Com cOrigem divergente: {len(divergentes_origem)}")

Títulos em comum (mesmo nCodTitulo nas duas fontes): 5108
Com cOrigem divergente: 0

**0 divergências em 5108 títulos** — `cOrigem` é idêntico nos dois endpoints pra
todo título comum. A heurística de religação/substituto/pagamento avulso se
comporta exatamente igual, não importa qual das duas fontes alimenta
`contratos.py`.

### 10.5 Item 4 — cobertura de teste offline com o shape de `PesquisarLancamentos`

`test_contratos_offline.py` só usava fixtures no formato de `ListarMovimentos`
(a função `_titulo()`, sem `aCodCateg`/`lancamentos`). Adicionei
`_titulo_pesquisarlancamentos()` (shape bruto real: `aCodCateg`, `lancamentos`,
`cCodCateg=None` quando rateado) e um novo teste,
`_testar_shape_pesquisarlancamentos()`, com dois cenários:

1. **Recomendado**: título bruto (sem expandir `aCodCateg`) direto em
   `montar_contratos` — funciona sem adaptador nenhum, porque a função nunca lê
   `aCodCateg`. Confirma que o único efeito de um título rateado é a categoria
   *exibida* do contrato ficar em branco (gap cosmético) quando ele é o título
   mais antigo por vencimento — o valor e a contagem de parcelas continuam
   corretos.
2. **Regressão documentada**: expandir `aCodCateg` antes de `montar_contratos`
   (o que o notebook fazia antes do fix do item 1) — confirma que isso quebra
   `total_parcelas`, pra não repetir o erro se alguém reconectar os adaptadores
   errados no futuro.

In [ ]:
!python ../tests/test_contratos_offline.py

OK: contrato com unica pendencia atrasada -> Em atraso, proxima_parcela preenchida
OK: contrato com unica parcela futura 'Em aberto' (sem Previsto) -> Ativo, nao Encerrado
OK: contrato 100% cancelado -> Cancelado
OK: contrato sem nada pendente ou previsto -> Encerrado (regressao: comportamento inalterado)
OK: contrato cadastrado sem nenhum titulo casado aparece no relatorio, status vem do cadastro
OK: titulo orfao com cliente/valor/data batendo com 1 contrato -> religado (vinculo=heuristico)
OK: titulo sem Ordem de Servico agora religa normalmente (exigencia de nCodOS removida)
OK: titulo com vencimento depois do fim da vigencia agora religa normalmente (teto de vigencia removido)
OK: titulo com valor fora da tolerancia continua nao religado (unico criterio que restou, alem de natureza/vigencia-inicial)
OK: titulo dentro do teto de plausibilidade vira sinal de titulos_para_revisao, com motivo explicando so o valor (vigencia nao e mais criterio)
OK: titulo MUITO diferente do valor do co

### Resultado — os 4 itens da lista de `nCodCtr` estão fechados

| Item | O que testava | Resultado |
|---|---|---|
| 1. Rateio dentro de contrato | Título de contrato com `aCodCateg` 2+ entradas quebra `total_parcelas`? | **Sim, quebrava** — corrigido com adaptador sem expansão (10.1), validado com caso sintético real (10.2) |
| 2. Religação heurística | Caminho `contratos_cadastro` (órfãos, substituto, pagamento avulso) funciona igual com PesquisarLancamentos? | **60/60 contratos idênticos**, incluindo `titulos_para_revisao` (10.3) |
| 3. Consistência de `cOrigem` | `MANR`/`VENR`/etc. batem título a título entre os dois endpoints? | **0 divergências em 5108 títulos comuns** (10.4) |
| 4. Cobertura offline | `test_contratos_offline.py` cobre o shape de `PesquisarLancamentos`? | **Novo teste adicionado** (`_testar_shape_pesquisarlancamentos`), cobrindo o uso recomendado e a regressão documentada (10.5) |

Nenhuma mudança em `src/contratos.py` foi necessária para nenhum dos 4 itens — só
o adaptador de entrada (seção 10.1) e o teste offline novo. `contratos.py` já era
correto e robusto o bastante pra aceitar `PesquisarLancamentos` como fonte, desde
que a entrada respeite a mesma premissa que sempre teve: 1 item da lista = 1
parcela inteira.